# Module 05: File Handling & OS

Learn how to read, write, and organize files — the foundation of every ML pipeline.

**ML Focus:**
- Reading datasets from CSV/JSON/JSONL files
- Saving and loading model artifacts
- Organizing project directories
- Cross-platform file operations with pathlib

## 1. Opening Files — The Right Way

Python's `open()` function is the gateway to all file I/O. The key habit: **always use a context manager** (`with` statement) so files close automatically.

In [ ]:
# Create a sample file to work with
with open('sample.txt', 'w') as f:
    f.write('Hello, file world!\n')
    f.write('Line 2: ML datasets live in files.\n')
    f.write('Line 3: Learn to read them.\n')

print('File created successfully')

# Read the file back
with open('sample.txt', 'r') as f:
    content = f.read()

print('--- Full content ---')
print(content)

# Context manager ensures f.close() is called automatically
print('File closed automatically after with block')

## 2. Reading Methods Compared

Three ways to read files. The right choice depends on file size and use case.

In [ ]:
# Method 1: read() — whole file at once
with open('sample.txt', 'r') as f:
    whole = f.read()
print('read() returns a single string:')
print(repr(whole[:50]), '...')
print()

# Method 2: readlines() — list of lines
with open('sample.txt', 'r') as f:
    lines = f.readlines()
print('readlines() returns a list:')
print(lines)
print()

# Method 3: iteration — memory efficient (best for large files)
print('Iterating line by line:')
with open('sample.txt', 'r') as f:
    for i, line in enumerate(f, 1):
        print('Line', i, ':', line.strip())

## 3. CSV Files — The Data Science Standard

CSV is the most common format for tabular data. Python's `csv` module handles parsing edge cases (quotes, commas in values).

In [ ]:
import csv

# Create a synthetic ML dataset as CSV
ml_data = [
    {'feature_1': 0.5, 'feature_2': 1.2, 'label': 'cat'},
    {'feature_1': 0.8, 'feature_2': 0.9, 'label': 'dog'},
    {'feature_1': 0.3, 'feature_2': 1.5, 'label': 'cat'},
    {'feature_1': 0.9, 'feature_2': 0.4, 'label': 'bird'},
    {'feature_1': 0.1, 'feature_2': 2.0, 'label': 'dog'},
]

fieldnames = ['feature_1', 'feature_2', 'label']

# Write CSV
with open('ml_dataset.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(ml_data)

print('CSV file written with', len(ml_data), 'rows')

# Read CSV back
loaded = []
with open('ml_dataset.csv', 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        loaded.append(row)

print('Read back', len(loaded), 'rows')
print('First row:', loaded[0])

## 4. JSON — Configuration and Metadata

JSON is ideal for structured data: model configs, hyperparameters, evaluation metrics.

In [ ]:
import json

# Model configuration
config = {
    'model_type': 'random_forest',
    'hyperparameters': {
        'n_estimators': 100,
        'max_depth': 10,
        'min_samples_split': 5
    },
    'dataset': 'iris',
    'version': 'v2.1'
}

# Write JSON with pretty formatting
with open('model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Config saved to model_config.json')

# Read it back
with open('model_config.json', 'r') as f:
    loaded_config = json.load(f)

print('Loaded config:')
print('  Model type:', loaded_config['model_type'])
print('  Estimators:', loaded_config['hyperparameters']['n_estimators'])

## 5. JSONL — The ML Workhorse

JSONL (JSON Lines) is the standard format for large ML datasets. Each line is one JSON object — you can append, stream, and process in parallel.

In [ ]:
import json

# Simulate a large ML dataset as JSONL
samples = []
for i in range(10):
    sample = {
        'id': i,
        'text': 'Sample text number ' + str(i),
        'label': 'positive' if i % 2 == 0 else 'negative',
        'features': [float(j) / 10.0 for j in range(5)]
    }
    samples.append(sample)

# Write JSONL (one JSON object per line)
with open('dataset.jsonl', 'w') as f:
    for s in samples:
        f.write(json.dumps(s) + '\n')

print('Wrote', len(samples), 'samples to dataset.jsonl')

# Read JSONL (memory efficient — process one line at a time)
loaded_samples = []
with open('dataset.jsonl', 'r') as f:
    for line in f:
        sample = json.loads(line.strip())
        loaded_samples.append(sample)

print('Loaded', len(loaded_samples), 'samples')
print('First sample id:', loaded_samples[0]['id'])
print('First sample label:', loaded_samples[0]['label'])

## 6. pathlib — Modern Path Management

`pathlib` is the modern, cross-platform way to handle file paths. It replaces `os.path` with an object-oriented API.

In [ ]:
from pathlib import Path
import os

# Path creation and joining
data_dir = Path('data')
csv_path = data_dir / 'raw' / 'train.csv'

print('Path:', csv_path)
print('Parent:', csv_path.parent)
print('Name:', csv_path.name)
print('Stem:', csv_path.stem)
print('Suffix:', csv_path.suffix)

# Create directories (like mkdir -p)
Path('models/linear_regression/v1').mkdir(parents=True, exist_ok=True)
print('Created directory tree: models/linear_regression/v1')

# Check existence
print('Data dir exists:', data_dir.exists())
print('models dir exists:', Path('models').exists())

# Current directory
print('\nCurrent directory:', Path.cwd())

## 7. Working with Directories

Listing, filtering, and organizing files in directories.

In [ ]:
from pathlib import Path

# Create some dummy files to work with
for name in ['data_01.csv', 'data_02.csv', 'model.pkl', 'config.json', 'logs.txt']:
    Path(name).touch()

# List all files
print('All files in current directory:')
for p in Path('.').iterdir():
    print(' ', p.name)

print()

# Glob pattern matching
print('CSV files:')
for p in Path('.').glob('*.csv'):
    print(' ', p.name)

print()
print('JSON files:')
for p in Path('.').glob('*.json'):
    print(' ', p.name)

## 8. File Metadata and Model Artifacts

In ML, you often need to check file sizes (is my dataset too big?), modification times (which model version is newest?), and organize artifacts.

In [ ]:
from pathlib import Path
import time
import json

# Check file metadata
p = Path('model_config.json')
stat = p.stat()

print('File:', p.name)
print('Size:', stat.st_size, 'bytes')
print('Modified:', time.ctime(stat.st_mtime))
print()

# Simulate saving a model with metadata
model_info = {
    'model_file': 'model_v1.pkl',
    'accuracy': 0.943,
    'features': 128,
    'dataset_size': 50000,
    'training_time_sec': 120.5
}

with open('model_v1_metadata.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print('Model metadata saved')
print('Artifacts organized: model file + metadata JSON')

## Summary

**Key takeaways:**
1. Always use `with open(...)` for file operations
2. CSV for tabular data, JSON for configs/metadata, JSONL for large ML datasets
3. Use `pathlib.Path` for cross-platform path operations
4. Organize model artifacts with companion metadata files
5. Check file existence and sizes before processing large datasets